# Alzheimer's Disease Progression Prediction using LSTM

This notebook predicts Alzheimer's disease diagnosis from longitudinal patient visit data using Bidirectional LSTM networks.  
The input data (`combined_imputed_4_visits.csv`) contains up to 4 chronologically ordered visits per patient with imputed clinical scores.

## 1 · Setup & Imports

Import all required libraries. GPU is auto-detected — if available it is used with memory-growth enabled to prevent OOM errors on local machines; otherwise training falls back to CPU.

In [ ]:
# ── Suppress noisy warnings ──────────────────────────────────────────────
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'          # hide TF info/warning logs

# ── Core ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Sklearn ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_fscore_support
)

# ── TensorFlow / Keras ───────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# ── GPU Setup (safe for local machines) ──────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ {len(gpus)} GPU(s) detected — using GPU")
else:
    print("ℹ️  No GPU detected — using CPU")

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Plot style ───────────────────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 2 · Load & Explore Data

Load the locally generated `combined_imputed_4_visits.csv`.  
This CSV was produced by `processing2.ipynb` which merged baseline/screening visits into a single *m0* row, selected each patient's earliest 4 visits, and imputed missing clinical scores using trajectory-aware strategies.

In [ ]:
df = pd.read_csv('combined_imputed_4_visits.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
# How many visits per patient?
visits_per_patient = df.groupby('subject_id').size()
print('Visits per patient distribution:')
print(visits_per_patient.value_counts().sort_index())
print(f'\nPatients with exactly 4 visits: {(visits_per_patient == 4).sum()}')
print(f'Total patients: {df["subject_id"].nunique()}')

## 3 · Data Preparation

Select the features that carry clinical progression signal.  
The target is `DIAGNOSIS` (1=CN, 2=MCI, 3=AD) — we remap to 0/1/2 for Keras.  
Patients with **any NaN** in the selected columns are dropped, and we keep only patients with **enough visits** for the experiment.

In [ ]:
FEATURES_BASE = ['entry_age', 'CDGLOBAL', 'MMSCORE', 'TOTSCORE']
FEATURES_EXT  = FEATURES_BASE + ['visit_month']   # extended set
TARGET = 'DIAGNOSIS'
LABEL_NAMES = ['CN', 'MCI', 'AD']
NUM_CLASSES = 3

# Sort chronologically within each patient
df = df.sort_values(['subject_id', 'visit_month']).reset_index(drop=True)

# Work with a clean subset
keep_cols = ['subject_id', 'visit_month'] + FEATURES_BASE + [TARGET]
df_model = df[keep_cols].copy()
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

# Remap target: (1,2,3) → (0,1,2)
df_model[TARGET] = df_model[TARGET].astype(int) - 1

print('Clean data shape:', df_model.shape)
print('\nClass distribution:')
print(df_model[TARGET].value_counts().rename(index=dict(enumerate(LABEL_NAMES))))

### Class Distribution Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df_model[TARGET].value_counts().sort_index()
bars = ax.bar(LABEL_NAMES, counts.values, color=['#2ecc71', '#f39c12', '#e74c3c'],
              edgecolor='white', linewidth=1.5)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v + 30, str(v),
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Diagnosis Class Distribution', fontweight='bold')
ax.set_ylabel('Number of visit records')
plt.tight_layout()
plt.savefig('plot_class_distribution.png', dpi=150)
plt.show()

## 4 · Patient-Level Train / Test Split

**Critical**: we split at the `subject_id` level so that all visits of a given patient go to either train or test — never both. This prevents data leakage and ensures honest evaluation.

In [ ]:
subjects = df_model['subject_id'].unique()
train_subj, test_subj = train_test_split(subjects, test_size=0.2, random_state=SEED)

train_df = df_model[df_model['subject_id'].isin(train_subj)].copy()
test_df  = df_model[df_model['subject_id'].isin(test_subj)].copy()

# Verify no overlap
assert len(set(train_subj) & set(test_subj)) == 0, 'LEAK: overlap between train/test patients!'

print(f'Train patients: {len(train_subj)} | Test patients: {len(test_subj)}')
print(f'Train rows: {len(train_df)} | Test rows: {len(test_df)}')

## 5 · Feature Scaling

MinMax-scale features to [0,1] for LSTM convergence.  
The scaler is **fit only on training data** to prevent information leakage.

In [ ]:
# We scale ALL candidate features; the experiment loop will select which ones to use.
ALL_FEATURES = FEATURES_EXT  # includes visit_month

scaler = MinMaxScaler()
train_df.loc[:, ALL_FEATURES] = scaler.fit_transform(train_df[ALL_FEATURES])
test_df.loc[:, ALL_FEATURES]  = scaler.transform(test_df[ALL_FEATURES])

print('Scaling done.  Feature ranges (train):')
train_df[ALL_FEATURES].describe().loc[['min','max']]

## 6 · Sequence Builder

Convert per-visit rows into fixed-length sequences for the LSTM.  
For a patient with visits [v1, v2, v3, v4] and `seq_len=3`:  
- Input: features of [v1, v2, v3] → Target: diagnosis at v4  

Patients with fewer than `seq_len + 1` visits are automatically skipped.

In [ ]:
def create_sequences(data, feature_cols, seq_len):
    """Build (X, y) sequences grouped by subject_id."""
    X, y = [], []
    for _, g in data.groupby('subject_id'):
        g = g.sort_values('visit_month')
        vals   = g[feature_cols].values
        labels = g[TARGET].values
        for i in range(len(vals) - seq_len):
            X.append(vals[i:i+seq_len])
            y.append(labels[i+seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

print('Sequence builder ready.')

## 7 · Model Builder

Parameterised model constructor for easy experimentation.  
Architecture: Bidirectional LSTM → BatchNorm → Dropout → Bidirectional LSTM → BatchNorm → Dropout → Dense (L2) → Softmax.

In [ ]:
def build_model(seq_len, n_features,
                lstm_units=64, dense_units=32,
                dropout=0.3, lr=0.001):
    model = Sequential([
        Bidirectional(
            LSTM(lstm_units, return_sequences=True),
            input_shape=(seq_len, n_features)
        ),
        BatchNormalization(),
        Dropout(dropout),

        Bidirectional(LSTM(lstm_units // 2)),
        BatchNormalization(),
        Dropout(dropout),

        Dense(dense_units, activation='relu',
              kernel_regularizer=l2(1e-4)),
        Dropout(dropout / 2),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print('Model builder ready.')

## 8 · Experiment: SEQ_LEN × Feature-Set Comparison

We train 4 configurations to find the best combination:

| # | SEQ_LEN | Features | Rationale |
|---|---------|----------|----------|
| A | 3 | base (4) | Original approach: 3 visits predict the 4th |
| B | 3 | +visit_month (5) | Same, but model knows *when* each visit occurred |
| C | 2 | base (4) | More training samples (2 per patient) |
| D | 2 | +visit_month (5) | More samples + temporal info |

Each is trained for 50 epochs with early stopping. The best is selected for full hyperparameter tuning.

In [ ]:
configs = [
    {'name': 'A  SEQ=3, base',        'seq_len': 3, 'features': FEATURES_BASE},
    {'name': 'B  SEQ=3, +visit_month', 'seq_len': 3, 'features': FEATURES_EXT},
    {'name': 'C  SEQ=2, base',        'seq_len': 2, 'features': FEATURES_BASE},
    {'name': 'D  SEQ=2, +visit_month', 'seq_len': 2, 'features': FEATURES_EXT},
]

experiment_results = []

for cfg in configs:
    name = cfg['name']
    sl   = cfg['seq_len']
    feat = cfg['features']
    print(f'\n{"="*60}')
    print(f'  Config: {name}')
    print(f'{"="*60}')

    X_tr, y_tr = create_sequences(train_df, feat, sl)
    X_te, y_te = create_sequences(test_df,  feat, sl)

    if len(X_tr) == 0 or len(X_te) == 0:
        print('  ⚠️  Not enough sequences — skipping')
        continue

    print(f'  Train samples: {len(X_tr)} | Test samples: {len(X_te)}')

    # Class weights
    cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
    cw_dict = dict(enumerate(cw))

    model = build_model(sl, len(feat), lstm_units=64, dense_units=32,
                        dropout=0.3, lr=0.001)

    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    rl = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=0)

    hist = model.fit(
        X_tr, y_tr,
        validation_split=0.2,
        epochs=50,
        batch_size=32,
        class_weight=cw_dict,
        callbacks=[es, rl],
        verbose=0
    )

    loss, acc = model.evaluate(X_te, y_te, verbose=0)
    print(f'  ➜ Test Accuracy: {acc:.4f}  |  Test Loss: {loss:.4f}')

    experiment_results.append({
        'config': name, 'seq_len': sl,
        'n_features': len(feat), 'features': feat,
        'train_samples': len(X_tr), 'test_samples': len(X_te),
        'test_acc': acc, 'test_loss': loss,
        'history': hist.history,
    })

### Experiment Results Comparison

In [ ]:
res_df = pd.DataFrame(experiment_results)[['config','seq_len','n_features',
                                            'train_samples','test_samples',
                                            'test_acc','test_loss']]
res_df = res_df.sort_values('test_acc', ascending=False).reset_index(drop=True)
print(res_df.to_string(index=False))

# Pick the best
best_exp = experiment_results[res_df.index[0]] if res_df.iloc[0]['test_acc'] == max(r['test_acc'] for r in experiment_results) else max(experiment_results, key=lambda x: x['test_acc'])
BEST_SEQ_LEN  = best_exp['seq_len']
BEST_FEATURES = best_exp['features']
print(f'\n✅ Best config: {best_exp["config"]}')
print(f'   SEQ_LEN={BEST_SEQ_LEN}, features={BEST_FEATURES}')

In [ ]:
# Bar chart comparing experiments
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']
names = [r['config'] for r in experiment_results]
accs  = [r['test_acc'] for r in experiment_results]
bars = ax.barh(names, accs, color=colors[:len(names)], edgecolor='white', height=0.5)
for b, v in zip(bars, accs):
    ax.text(v + 0.003, b.get_y() + b.get_height()/2,
            f'{v:.4f}', va='center', fontweight='bold')
ax.set_xlim(0, 1.0)
ax.set_xlabel('Test Accuracy')
ax.set_title('Experiment Comparison: SEQ_LEN × Feature Set', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_experiment_comparison.png', dpi=150)
plt.show()

## 9 · Hyperparameter Tuning (on Best Config)

We perform a grid search over LSTM units, dense units, dropout rate, and learning rate using the winning configuration from the experiment above. Each combo trains for 30 epochs with early stopping.

In [ ]:
# Build sequences for the best config
X_train, y_train = create_sequences(train_df, BEST_FEATURES, BEST_SEQ_LEN)
X_test,  y_test  = create_sequences(test_df,  BEST_FEATURES, BEST_SEQ_LEN)

print(f'Tuning on: SEQ_LEN={BEST_SEQ_LEN}, features={len(BEST_FEATURES)}')
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

# Class weights
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))
print('Class weights:', {LABEL_NAMES[k]: f'{v:.3f}' for k, v in class_weights.items()})

In [ ]:
param_grid = {
    'lstm_units':   [64, 128],
    'dense_units':  [32, 64],
    'dropout':      [0.2, 0.35],
    'lr':           [0.001, 0.0005],
}

tuning_results = []
best_acc  = 0
best_params = None

total = 1
for v in param_grid.values():
    total *= len(v)
print(f'Grid search: {total} configurations\n')

run = 0
for lstm_u in param_grid['lstm_units']:
    for dense_u in param_grid['dense_units']:
        for drop in param_grid['dropout']:
            for lr in param_grid['lr']:
                run += 1
                print(f'[{run}/{total}] LSTM={lstm_u} Dense={dense_u} '
                      f'Drop={drop} LR={lr}', end=' ')

                m = build_model(BEST_SEQ_LEN, len(BEST_FEATURES),
                                lstm_u, dense_u, drop, lr)

                es = EarlyStopping(monitor='val_loss', patience=8,
                                   restore_best_weights=True)

                m.fit(X_train, y_train,
                      validation_split=0.2,
                      epochs=30, batch_size=32,
                      class_weight=class_weights,
                      callbacks=[es], verbose=0)

                _, acc = m.evaluate(X_test, y_test, verbose=0)
                print(f'→ acc={acc:.4f}')

                tuning_results.append({
                    'lstm_units': lstm_u, 'dense_units': dense_u,
                    'dropout': drop, 'lr': lr, 'accuracy': acc
                })

                if acc > best_acc:
                    best_acc = acc
                    best_params = dict(lstm_units=lstm_u, dense_units=dense_u,
                                      dropout=drop, lr=lr)

print(f'\n✅ Best grid-search accuracy: {best_acc:.4f}')
print(f'   Params: {best_params}')

In [ ]:
tune_df = pd.DataFrame(tuning_results).sort_values('accuracy', ascending=False)
tune_df.head(10)

## 10 · Final Model Training

Retrain the best configuration for up to 150 epochs with early stopping (patience=15) and learning rate reduction to squeeze out the best performance.

In [ ]:
final_model = build_model(
    BEST_SEQ_LEN, len(BEST_FEATURES), **best_params
)
final_model.summary()

In [ ]:
es_final = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
rl_final = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

history = final_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[es_final, rl_final],
    verbose=1
)

## 11 · Training Curves

Loss and accuracy over epochs for both training and validation sets.  
A small gap between the curves indicates good generalisation; divergence signals overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history['loss'],     label='Train Loss', linewidth=2)
ax1.plot(history.history['val_loss'],  label='Val Loss',   linewidth=2, linestyle='--')
ax1.set_title('Loss over Epochs', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history['accuracy'],     label='Train Acc', linewidth=2)
ax2.plot(history.history['val_accuracy'],  label='Val Acc',   linewidth=2, linestyle='--')
ax2.set_title('Accuracy over Epochs', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plot_training_curves.png', dpi=150)
plt.show()

## 12 · Evaluation on Test Set

In [ ]:
test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)
print(f'Final Test Loss:     {test_loss:.4f}')
print(f'Final Test Accuracy: {test_acc:.4f}')

y_pred_probs = final_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

### Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES, digits=4))

### Confusion Matrix

Rows = true class, columns = predicted class.  
Ideally values concentrate on the diagonal.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            linewidths=1, linecolor='white', ax=ax)
ax.set_xlabel('Predicted', fontweight='bold')
ax.set_ylabel('Actual', fontweight='bold')
ax.set_title('Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_confusion_matrix.png', dpi=150)
plt.show()

### ROC Curves (One-vs-Rest)

Shows the trade-off between true-positive rate and false-positive rate for each class.  
AUC closer to 1.0 indicates better discriminative ability.

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(7, 6))
colors_roc = ['#2ecc71', '#f39c12', '#e74c3c']

for i, (label, color) in enumerate(zip(LABEL_NAMES, colors_roc)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{label}  (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (One-vs-Rest)', fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_roc_curves.png', dpi=150)
plt.show()

### Per-Class Precision, Recall & F1-Score

In [ ]:
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0,1,2])

x_pos = np.arange(NUM_CLASSES)
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_pos - width, prec, width, label='Precision', color='#3498db')
ax.bar(x_pos,         rec,  width, label='Recall',    color='#2ecc71')
ax.bar(x_pos + width, f1,   width, label='F1-Score',  color='#e74c3c')

ax.set_xticks(x_pos)
ax.set_xticklabels(LABEL_NAMES)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Per-Class Precision · Recall · F1', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Value labels
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=9)

plt.tight_layout()
plt.savefig('plot_per_class_metrics.png', dpi=150)
plt.show()

### Actual vs Predicted — Sample Comparison

A side-by-side look at the first 80 test predictions to visually inspect agreement.

In [ ]:
N_SHOW = min(80, len(y_test))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(N_SHOW), y_test[:N_SHOW],  marker='o', markersize=4,
        label='Actual', linewidth=1, alpha=0.8)
ax.plot(range(N_SHOW), y_pred[:N_SHOW],  marker='x', markersize=5,
        label='Predicted', linewidth=1, linestyle='--', alpha=0.8)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(LABEL_NAMES)
ax.set_xlabel('Test Sample Index')
ax.set_title('Actual vs Predicted Diagnosis (first 80 samples)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_actual_vs_predicted.png', dpi=150)
plt.show()

### Prediction Confidence Distribution

Histogram of the model's confidence (max softmax probability) for correct vs incorrect predictions.  
Good models are highly confident when correct and less confident when wrong.

In [ ]:
max_probs = y_pred_probs.max(axis=1)
correct = (y_pred == y_test)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(max_probs[correct],  bins=30, alpha=0.7, label='Correct',   color='#2ecc71')
ax.hist(max_probs[~correct], bins=30, alpha=0.7, label='Incorrect', color='#e74c3c')
ax.set_xlabel('Max Softmax Probability (Confidence)')
ax.set_ylabel('Count')
ax.set_title('Prediction Confidence Distribution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_confidence_distribution.png', dpi=150)
plt.show()

## 13 · Summary

Print a consolidated summary of the best configuration and final metrics.

In [ ]:
print('=' * 60)
print('         FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'Best Configuration : {best_exp["config"]}')
print(f'Sequence Length    : {BEST_SEQ_LEN}')
print(f'Features           : {BEST_FEATURES}')
print(f'Hyperparameters    : {best_params}')
print(f'Train samples      : {len(X_train)}')
print(f'Test samples       : {len(X_test)}')
print(f'Test Accuracy      : {test_acc:.4f}')
print(f'Test Loss          : {test_loss:.4f}')
print('=' * 60)
print()
print('Plots saved:')
for f in sorted([f for f in os.listdir('.') if f.startswith('plot_') and f.endswith('.png')]):
    print(f'  • {f}')